# Task 2: Baseline Models

Four baselines, each evaluated and saved to `submissions/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from collections import Counter

from src import (
    load_articles, load_categories, load_states_train, load_states_test,
    load_or_build_adjacency, make_submission, validate_submission
)

DATA = Path('dataset-task2')
CACHE = Path('.cache')

articles = load_articles()
categories = load_categories()
train = load_states_train()
test = load_states_test()
adj = load_or_build_adjacency()

# Load precomputed embeddings
embeddings = np.load(CACHE / 'article_embeddings.npy')

all_cats = sorted(categories['category'].unique())
cat_to_idx = {c: i for i, c in enumerate(all_cats)}
cat_enc = np.zeros((len(articles), len(all_cats)), dtype=np.float32)
for _, r in categories.iterrows():
    cat_enc[r['article_id'], cat_to_idx[r['category']]] = 1.0

print(f'Loaded {len(articles)} articles, {len(train)} train, {len(test)} test states')

## Baseline 1: Title Similarity

Pick the link on current page whose title embedding is most similar (cosine) to the target article's embedding.

In [ ]:
def predict_title_similarity(states_df, adj, embeddings):
    preds = []
    for _, r in tqdm(states_df.iterrows(), total=len(states_df), desc='Title sim'):
        links = adj.get(r['current_article_id'], [])
        if not links:
            preds.append(r['target_article_id'])
            continue
        scores = embeddings[links] @ embeddings[r['target_article_id']]
        preds.append(links[scores.argmax()])
    return np.array(preds)

train_preds = predict_title_similarity(train, adj, embeddings)
train_acc = (train_preds == train['next_article_id'].values).mean()
print(f'Train accuracy: {train_acc*100:.2f}%')

test_preds = predict_title_similarity(test, adj, embeddings)
sub_dir = Path('submissions') / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_title_sim'
sub_dir.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, test_preds, sub_dir / 'submission.csv')
validate_submission(sub_dir / 'submission.csv')
print(f'Saved: {sub_dir / "submission.csv"}')

## Baseline 2: Category Overlap

Pick the link sharing the most categories with the target.

In [ ]:
def predict_category_overlap(states_df, adj, cat_enc):
    preds = []
    for _, r in tqdm(states_df.iterrows(), total=len(states_df), desc='Category'):
        links = adj.get(r['current_article_id'], [])
        if not links:
            preds.append(r['target_article_id'])
            continue
        link_cats = cat_enc[links]
        tgt_cats = cat_enc[r['target_article_id']][None, :]
        scores = (link_cats * tgt_cats).sum(axis=1)
        preds.append(links[scores.argmax()])
    return np.array(preds)

train_preds = predict_category_overlap(train, adj, cat_enc)
train_acc = (train_preds == train['next_article_id'].values).mean()
print(f'Train accuracy: {train_acc*100:.2f}%')

test_preds = predict_category_overlap(test, adj, cat_enc)
sub_dir = Path('submissions') / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_category'
sub_dir.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, test_preds, sub_dir / 'submission.csv')
validate_submission(sub_dir / 'submission.csv')
print(f'Saved: {sub_dir / "submission.csv"}')

## Baseline 3: Most Popular Link

For each current article, pick the link most frequently clicked as next_article in training.

In [ ]:
# Build popularity: for each current article, count how often each next_article appears
popularity = {}
for _, r in train.iterrows():
    curr = r['current_article_id']
    nxt = r['next_article_id']
    if curr not in popularity:
        popularity[curr] = Counter()
    popularity[curr][nxt] += 1

# Most popular per current
most_popular = {}
for curr, counter in popularity.items():
    most_popular[curr] = counter.most_common(1)[0][0]

def predict_popular(states_df, adj, popularity):
    preds = []
    for _, r in tqdm(states_df.iterrows(), total=len(states_df), desc='Popular'):
        curr = r['current_article_id']
        if curr in popularity:
            preds.append(popularity[curr].most_common(1)[0][0])
        else:
            links = adj.get(curr, [])
            preds.append(links[0] if links else r['target_article_id'])
    return np.array(preds)

train_preds = predict_popular(train, adj, popularity)
train_acc = (train_preds == train['next_article_id'].values).mean()
print(f'Train accuracy: {train_acc*100:.2f}%')

test_preds = predict_popular(test, adj, popularity)
sub_dir = Path('submissions') / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_popular'
sub_dir.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, test_preds, sub_dir / 'submission.csv')
validate_submission(sub_dir / 'submission.csv')
print(f'Saved: {sub_dir / "submission.csv"}')

## Baseline 4: XGBoost

Train a binary classifier on (state, candidate) pairs to predict if a candidate is the next click.
Then at inference: score all candidates for a state, pick the highest.

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split

with open(CACHE / 'xgb_features.pkl', 'rb') as f:
    feat_data = pickle.load(f)

X = feat_data['X']
y = feat_data['y']
info = feat_data['info']

# Split: 80/20 train/val by state_id (group level to avoid leakage)
state_ids = list(set(s['state_id'] for s in info))
train_ids, val_ids = train_test_split(state_ids, test_size=0.2, random_state=42)
train_ids = set(train_ids)

train_mask = [s['state_id'] in train_ids for s in info]
val_mask = [s['state_id'] not in train_ids for s in info]

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Train: {len(X_train)}, Val: {len(X_val)}, pos_weight: {scale_pos_weight:.1f}')

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42, n_jobs=-1
)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)

In [ ]:
def predict_xgboost(states_df, adj, embeddings, cat_enc, model):
    preds = []
    for _, r in tqdm(states_df.iterrows(), total=len(states_df), desc='XGBoost'):
        curr, tgt = r['current_article_id'], r['target_article_id']
        links = adj.get(curr, [])
        if not links:
            preds.append(tgt)
            continue
        curr_emb, tgt_emb = embeddings[curr], embeddings[tgt]
        curr_cats, tgt_cats = cat_enc[curr], cat_enc[tgt]
        
        feat_list = []
        for link in links:
            link_emb = embeddings[link]
            link_cats = cat_enc[link]
            feat_list.append([
                float(curr_emb @ tgt_emb),
                float(link_emb @ tgt_emb),
                float(link_emb @ curr_emb),
                int((curr_cats * tgt_cats).sum()),
                int((link_cats * tgt_cats).sum()),
                len(adj.get(link, [])),
                len(links),
                int(link == tgt),
                int(link == curr),
                float(curr_emb @ link_emb - curr_emb @ tgt_emb),
            ])
        scores = model.predict_proba(np.array(feat_list))[:, 1]
        preds.append(links[scores.argmax()])
    return np.array(preds)

train_preds = predict_xgboost(train, adj, embeddings, cat_enc, model)
train_acc = (train_preds == train['next_article_id'].values).mean()
print(f'Train accuracy: {train_acc*100:.2f}%')

test_preds = predict_xgboost(test, adj, embeddings, cat_enc, model)
sub_dir = Path('submissions') / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_xgb'
sub_dir.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, test_preds, sub_dir / 'submission.csv')
model.save_model(str(sub_dir / 'xgb_model.json'))
validate_submission(sub_dir / 'submission.csv')
print(f'Saved: {sub_dir / "submission.csv"}')

## Baseline 5: Ensemble (majority vote)

Combine predictions from all baselines via majority vote.

In [ ]:
def predict_ensemble(states_df, predictors):
    all_preds = []
    for _, r in tqdm(states_df.iterrows(), total=len(states_df), desc='Ensemble'):
        votes = []
        for predictor in predictors:
            pred = predictor(r)
            votes.append(pred)
        counter = Counter(votes)
        all_preds.append(counter.most_common(1)[0][0])
    return np.array(all_preds)

# Build predictors
def make_title_pred(r):
    links = adj.get(r['current_article_id'], [])
    if not links: return r['target_article_id']
    scores = embeddings[links] @ embeddings[r['target_article_id']]
    return links[scores.argmax()]

def make_cat_pred(r):
    links = adj.get(r['current_article_id'], [])
    if not links: return r['target_article_id']
    scores = (cat_enc[links] * cat_enc[r['target_article_id']][None, :]).sum(axis=1)
    return links[scores.argmax()]

def make_popular_pred(r):
    curr = r['current_article_id']
    if curr in popularity:
        return popularity[curr].most_common(1)[0][0]
    links = adj.get(curr, [])
    return links[0] if links else r['target_article_id']

predictors = [make_title_pred, make_cat_pred, make_popular_pred]

train_preds = predict_ensemble(train, predictors)
train_acc = (train_preds == train['next_article_id'].values).mean()
print(f'Train ensemble accuracy: {train_acc*100:.2f}%')

test_preds = predict_ensemble(test, predictors)
sub_dir = Path('submissions') / f'{datetime.now().strftime("%Y%m%d_%H%M%S")}_ensemble_vote'
sub_dir.mkdir(parents=True, exist_ok=True)
make_submission(test['state_id'].values, test_preds, sub_dir / 'submission.csv')
validate_submission(sub_dir / 'submission.csv')
print(f'Saved: {sub_dir / "submission.csv"}')

## Summary

In [ ]:
print('Baselines complete.')
print('Submissions saved in submissions/<timestamp>_<model>/')